# Bagging - кастомная реализация

In [6]:
# Установка зависимостей
# Установка зависимостей
!pip install -q \
  scikit-learn==1.6.1 \
  numpy \
  pandas

In [7]:
# Импортируем необходимые библиотеки
import numpy as np
from sklearn.tree import DecisionTreeClassifier
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

# Создаём синтетические данные для классификации
X, y = make_classification(n_samples=15000, n_features=25, n_classes=2, random_state=42, n_informative=20)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Параметры бэггинга
n_trees = 23 
max_depth = 5

print(X_train.shape)
print(X_test.shape)

(10500, 25)
(4500, 25)


In [10]:
# Реализуем бэггинг
from sklearn.tree import DecisionTreeClassifier

trees = []
for _ in range(n_trees):
    # Формируем бутстрап-выборку (с возвращением)
    indices = np.random.choice(len(X_train), size=len(X_train), replace=True)
    X_bootstrap = X_train[indices]
    y_bootstrap = y_train[indices]

    # Обучим дерево на бутстрап-выборке
    tree = DecisionTreeClassifier(max_depth=max_depth, random_state=42)
    tree.fit(X_bootstrap, y_bootstrap)
    trees.append(tree)

len(trees) # Печатаем массив с деревьями

23

In [11]:
# Предсказания ансамбля (голосование для классификации)

def predict_ensemble(X):
    predictions = np.array([tree.predict(X) for tree in trees])
    # Голосование по большинству
    return np.round(predictions.mean(axis=0)).astype(int) # Для двух классов усреднение и округление работает как голосование (для многоклассовых задач нужно использовать argmax)

# Оцениваем точность

y_pred = predict_ensemble(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"Точность ансамбля: {accuracy:.4f}")

# Строим отчёт по всем метрикам
report = classification_report(y_test, y_pred)
print(report)

Точность ансамбля: 0.8220
              precision    recall  f1-score   support

           0       0.87      0.75      0.81      2249
           1       0.78      0.89      0.83      2251

    accuracy                           0.82      4500
   macro avg       0.83      0.82      0.82      4500
weighted avg       0.83      0.82      0.82      4500



In [15]:
# Одно дерево на всей выборке
solo_tree = DecisionTreeClassifier(max_depth=max_depth, random_state=42)
solo_tree.fit(X_train, y_train)

y_pred_solo_tree = solo_tree.predict(X_test)

# Точность одного дерева
solo_tree_accuracy = accuracy_score(y_test, y_pred_solo_tree)
print(f"Точность одного дерева: {solo_tree_accuracy:.4f}")

# Отчёт по всем метрикам
solo_tree_report = classification_report(y_test, y_pred_solo_tree)
print(solo_tree_report)

Точность одного дерева: 0.7751
              precision    recall  f1-score   support

           0       0.80      0.73      0.76      2249
           1       0.75      0.82      0.78      2251

    accuracy                           0.78      4500
   macro avg       0.78      0.78      0.77      4500
weighted avg       0.78      0.78      0.77      4500



Объединение моделей в ансамбль дало прирост в точности и позволило лучше обработать граничные случаи

Одна модель может быть подвержена переобучению или недообучению, тогда как ансамбль снижает дисперсию и компенсирует ошибки отдельных моделей за счёт усреднения или голосования. Правильный ответ
Ансамбль комбинирует предсказания нескольких моделей, обученных на разных подвыборках данных, что позволяет снизить влияние шума и случайных колебаний